# Estructuras de Datos en Python — Nivel Difícil 🌳

**¿Cómo elegir tu nivel?** Este es uno de tres notebooks de práctica (fácil, medio, difícil). Este nivel es para ti si ya tienes experiencia previa programando (en Python o en otro lenguaje) y quieres ir directo a retos con estructuras anidadas y funciones. No hay ninguna jerarquía entre los niveles — elige el que te haga aprender más hoy, no el que "debas" elegir.

Vamos a trabajar con **registros de pacientes de una clínica**, usando estructuras anidadas (diccionarios dentro de diccionarios, listas dentro de diccionarios) — muy poco andamiaje, más retos abiertos.


## 1. Variables primitivas: casting y valores faltantes

Repaso exprés: `int`, `float`, `str`, `bool`, casting con `int(...)`/`float(...)`. Un problema muy común en datos reales: valores **faltantes**. En Python puro, `None` representa "no hay dato" (más adelante, Pandas usará `NaN` para lo mismo).

In [ ]:
temperatura_texto = "38.2"
presion_texto = None  # este paciente no tiene el dato registrado

temperatura = float(temperatura_texto)
presion = float(presion_texto) if presion_texto is not None else None

print(temperatura, type(temperatura))
print(presion, type(presion))


### ✏️ Ejercicio 1

Escribe una función `a_float_seguro(valor)` que reciba un valor (puede ser un string numérico o `None`) y devuelva el `float` correspondiente, o `None` si el valor de entrada es `None`. Pruébala con `"36.5"` y con `None`.

In [ ]:
def a_float_seguro(valor):
    if valor is None:
        return None
    return float(valor)


print(a_float_seguro("36.5"))
print(a_float_seguro(None))


**Solución:** (la de arriba ya es una solución válida; también es correcto usar `try/except` si prefieres ese estilo).

## 2. Strings: limpieza de texto compuesto

Un campo de texto real casi nunca viene limpio. Aquí tienes una lista de síntomas separada por comas, con mayúsculas/espacios inconsistentes. Vamos a limpiarla con una *list comprehension*.

In [ ]:
sintomas_crudos = "  Fiebre,Tos, dolor de cabeza ,NAUSEAS"

sintomas = [s.strip().capitalize() for s in sintomas_crudos.split(",")]
print(sintomas)


### ✏️ Ejercicio 2

Escribe una función `parsear_sintomas(texto)` que reciba un string como `sintomas_crudos` y devuelva la lista de síntomas limpios (usa la misma lógica de arriba, pero empaquetada en una función). Pruébala con al menos dos strings distintos, incluyendo uno con síntomas repetidos (la función no necesita eliminar duplicados, solo limpiar).

In [ ]:
def parsear_sintomas(texto):
    return [s.strip().capitalize() for s in texto.split(",")]


print(parsear_sintomas("  Fiebre,Tos, dolor de cabeza ,NAUSEAS"))
print(parsear_sintomas("tos,TOS , Fiebre"))


**Solución:** ya está arriba — compárala con tu propia implementación.

## 3. Listas de diccionarios anidados

Aquí está nuestro dataset: una lista de pacientes, cada uno con **signos vitales anidados** (un diccionario dentro del diccionario) y una **lista de diagnósticos**.

In [ ]:
pacientes = [
    {
        "nombre": "  maría lopez",
        "edad": 34,
        "signos_vitales": {"temperatura": 38.6, "presion": 118},
        "diagnosticos": ["Gripe", "Deshidratación"],
    },
    {
        "nombre": "CARLOS Ruiz",
        "edad": 58,
        "signos_vitales": {"temperatura": 36.9, "presion": 145},
        "diagnosticos": ["Hipertensión"],
    },
    {
        "nombre": "ana Torres ",
        "edad": 27,
        "signos_vitales": {"temperatura": 37.1, "presion": None},
        "diagnosticos": [],
    },
    {
        "nombre": "Jose Ramirez",
        "edad": 71,
        "signos_vitales": {"temperatura": 39.0, "presion": 160},
        "diagnosticos": ["Neumonía", "Hipertensión", "Fiebre alta"],
    },
]

edades = [p["edad"] for p in pacientes]
print("Edad promedio:", round(sum(edades) / len(edades), 1))

nombres_con_multiples_diagnosticos = [p["nombre"].strip().title() for p in pacientes if len(p["diagnosticos"]) > 1]
print("Con más de un diagnóstico:", nombres_con_multiples_diagnosticos)


### ✏️ Ejercicio 3

Escribe una función `resumen_paciente(paciente)` que reciba un diccionario de paciente (con la misma forma que arriba) y devuelva un string con este formato:

```
María Lopez (34 años): temperatura 38.6°C -> FIEBRE, 2 diagnóstico(s)
```

Reglas:
- El nombre debe quedar limpio (`.strip().title()`).
- Si `temperatura` es mayor a 38.0, agrega `-> FIEBRE`; si no, agrega `-> normal`.
- Si `temperatura` es `None` (no lo hay en este dataset, pero tu función debe soportarlo), muestra `sin dato` en vez de fallar.

In [ ]:
def resumen_paciente(paciente):
    nombre = paciente["nombre"].strip().title()
    edad = paciente["edad"]
    temperatura = paciente["signos_vitales"]["temperatura"]
    n_diagnosticos = len(paciente["diagnosticos"])

    if temperatura is None:
        estado_temp = "sin dato"
    elif temperatura > 38.0:
        estado_temp = f"temperatura {temperatura}°C -> FIEBRE"
    else:
        estado_temp = f"temperatura {temperatura}°C -> normal"

    return f"{nombre} ({edad} años): {estado_temp}, {n_diagnosticos} diagnóstico(s)"


for p in pacientes:
    print(resumen_paciente(p))


**Solución:** ya está arriba. Si tu versión da el mismo resultado con una estructura distinta (por ejemplo usando `if/elif/else` en vez de mi orden), ¡también está bien!

## 4. Diccionarios: construir un reporte agregado

Ahora vamos a agregar información de **todos** los pacientes en un solo diccionario-reporte, algo muy parecido a lo que hace un `.describe()` o un `.groupby()` de Pandas, pero escrito a mano.

In [ ]:
def construir_reporte(pacientes):
    edades = [p["edad"] for p in pacientes]
    temperaturas = [p["signos_vitales"]["temperatura"] for p in pacientes if p["signos_vitales"]["temperatura"] is not None]
    con_fiebre = [p for p in pacientes if p["signos_vitales"]["temperatura"] is not None and p["signos_vitales"]["temperatura"] > 38.0]

    conteo_diagnosticos = {}
    for p in pacientes:
        for d in p["diagnosticos"]:
            conteo_diagnosticos[d] = conteo_diagnosticos.get(d, 0) + 1

    return {
        "total_pacientes": len(pacientes),
        "edad_promedio": round(sum(edades) / len(edades), 1),
        "porcentaje_con_fiebre": round(100 * len(con_fiebre) / len(pacientes), 1),
        "conteo_diagnosticos": conteo_diagnosticos,
    }


reporte = construir_reporte(pacientes)
for llave, valor in reporte.items():
    print(f"{llave}: {valor}")


### ✏️ Reto final (opcional)

Extiende `construir_reporte` (o escribe una nueva función `diagnostico_mas_comun(pacientes)`) que devuelva el nombre del diagnóstico que más se repite entre todos los pacientes, junto con cuántas veces aparece. Pista: ya tienes `conteo_diagnosticos` calculado arriba — solo necesitas encontrar la llave con el valor máximo (puedes usar `max(diccionario, key=diccionario.get)`).

In [ ]:
def diagnostico_mas_comun(pacientes):
    conteo = {}
    for p in pacientes:
        for d in p["diagnosticos"]:
            conteo[d] = conteo.get(d, 0) + 1

    if not conteo:
        return None, 0

    diagnostico_top = max(conteo, key=conteo.get)
    return diagnostico_top, conteo[diagnostico_top]


nombre_dx, veces = diagnostico_mas_comun(pacientes)
print(f"Diagnóstico más común: {nombre_dx} ({veces} pacientes)")


¡Excelente trabajo! Acabas de construir a mano varias cosas que Pandas hace en una línea: agregaciones, conteos por categoría, y manejo de datos anidados/faltantes. Eso es exactamente lo que vas a ver mecanizado (y aplicado a datasets mucho más grandes) en la próxima sesión.